# The Empathetic Fitting: Room Real-Time Stress-Adaptive Controller

This notebook contains the system that drives the physiologically adaptive retail environment described in the dissertation. It listens to live RR-interval data
from the Polar H10 sensor over BLE, calculates a continuous stress/calm score using a PID controller, and uses that score to produce two simultaneous outputs:

- **Ambient lighting** (brightness and colour temperature) via a Hue BLE light
- **Dynamic music** (four audio stems that fade in/out) via `pygame.mixer`

Every heartbeat is also logged to CSV for later statistical analysis, and a live JSON file is written out for a separate live monitor interface.

**Pipeline overview:**

1. Load and loop four audio stems at zero volume
2. Calibrate a personal baseline RR-interval from the first 20 realistic heartbeats
3. On every subsequent heartbeat: compute a smoothed RR average, derive a stress score, run it through a PID controller, translate the result into light + music targets, move the light gradually toward the target, update music volumes, and log everything


## 1. Imports

In [1]:
import asyncio # To allow for multiple simultaneous actions
import time # To measure time between heartbeats
import pygame # To play/control music
import csv
import os
import json
from bleak import BleakClient, BleakScanner # To connect to Bluetooth devices
from bleakheart import HeartRate # To read heartbeat from Polar
from HueBLE import HueBleLight # To control lighs

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


## 2. Live Monitor File (JSON)

The main program and the live monitor run independently. To share data, the main program saves its current state to a JSON file (`monitor_data.json`) on every heartbeat, which the monitor then reads. If a file update temporarily fails, the system safely ignores the error.

In [2]:
monitor_file = "monitor_data.json"

def write_monitor_data(data_dict):
    """Saves the latest data to a JSON file for the live monitor to read."""
    try:
        with open(monitor_file, "w") as f:
            json.dump(data_dict, f) # .dump saves dictionary to JSON
    except Exception:
        pass # If the write fails, skip it

## 3. Session Logging (CSV)

Every heartbeat is saved as a new row in `session_log.csv`, which provides the data for the final statistical analysis. A new file is created for each session. For every heartbeat, the log records the time, heart rate data, PID score, and the resulting light and music settings. It also logs the cycle time (how many ms the system took to process that beat) to prove the software reacts in real time.

In [3]:
log_file = "session_log.csv"

log_headers = [
    "time_seconds",
    "rr_avg",
    "baseline_rr",
    "difference",
    "reaction_score",
    "brightness",
    "colour_temp",
    "vol_drone",
    "vol_chords",
    "vol_rhythm",
    "vol_melody",
    "cycle_time_ms",
]

def start_log():
    """Creates a new CSV log file at the start of each session. If a file already exists from a previous session, it gets overwritten."""
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(log_headers) # Writing headers first
    print("Session log started.\n") # Saved in session_log.csv

def write_log_row(time_s, rr_avg, baseline, diff, score, brightness, ct, v1, v2, v3, v4, cycle_ms):
    """Takes all the data generated from a single heartbeat and adds it as a new row."""
    with open(log_file, "a", newline="") as f: # a: to append row at the bottom
        writer = csv.writer(f)
        writer.writerow([
            round(time_s, 2),
            round(rr_avg, 1),
            round(baseline, 1),
            round(diff, 1),
            round(score, 2),
            brightness,
            ct,
            round(v1, 3),
            round(v2, 3),
            round(v3, 3),
            round(v4, 3),
            round(cycle_ms, 2),
        ])


## 4. Music Set-Up

The system loads four audio tracks (drone, chords, rhythm, melody) that loop, initially silently, in the background. Instead of switching tracks, it acts like a live mixing desk, fading these individual layers in one by one as the participant calms down.

In [4]:
pygame.mixer.init() # Initialising pygame

print("Loading audio stems...")

try:
    layer1_drone  = pygame.mixer.Sound("stem_1_drone.wav") # pygame.mixer.Sound is used to play different, short loops contemporarily
    layer2_chords = pygame.mixer.Sound("chords_24s.wav")
    layer3_rhythm = pygame.mixer.Sound("rhythm_24s.wav")
    layer4_melody = pygame.mixer.Sound("stem_4_melody.wav")
except FileNotFoundError:
    print("Error: One or more audio files were not found.")
    exit() # Ensures execution stops if files aren't found

# Starts playing all 4 tracks immediately to ensure they stay in sync, looping forever (-1 is used to loop forever until .stop())
ch1 = layer1_drone.play(-1)
ch1.set_volume(0.0) # Silent (0.0) until calibration finishes

ch2 = layer2_chords.play(-1)
ch2.set_volume(0.0)

ch3 = layer3_rhythm.play(-1)
ch3.set_volume(0.0)

ch4 = layer4_melody.play(-1)
ch4.set_volume(0.0)

print("Audio loaded and ready.\n")

Loading audio stems...
Audio loaded and ready.



## 5. Configuration Constants

These are the adjustable parameters of the system, grouped together so that they can be easily adjusted during pilot session:

- `beats_number_avg`: How many heartbeats to use for smoothing data.
- `calibration_beats`: How many heartbeats to use for setting the baseline.
- `KP`, `KI`, `KD`: The values that decide how quickly and strongly the system reacts to changes in the user's stress.
- `PID_min` / `PID_max`: The limits of the reaction score (-100 = very stressed, +100 = very calm).
- Light targets at each extreme (`brightness_calm/stressed`, `colour_temp_calm/stressed`)
- `rr_sensitivity`: This defines defines how much of a biological change is required to push the system to its limits. This was established during testing, as the sensitivity boundary of 150 ms proved to provide the best user experience. If the value was set too low, the room would instantly shift to maximum brightness at the slightest movement. If it was set too high, the system would feel unresponsive.

In [ ]:
beats_number_avg = 8 # The more beats, the smoother the system's reaction but the slower to respond to changes
calibration_beats = 20

KP = 0.8
KI = 0.05
KD = 0.3

PID_min = -100.0
PID_max = 100.0

brightness_calm = 254   
colour_temp_calm = 153   

brightness_stressed = 1
colour_temp_stressed = 500 

rr_sensitivity = 150.0

## 6. Dynamic State Variables

These variables define the default calm starting values, to serve as a baseline and initial reference. During a live session, the actual data is managed by the `StressController` and `LightController`.

In [6]:
pid_accumulated_error = 0.0
pid_last_stress_level = 0.0
pid_last_timestamp = None

current_brightness = brightness_calm
current_colour_temp = colour_temp_calm

## 7. Helper Functions and Classes

### 7.1 Sensor data validation

The sensor might sometimes receive bad data due to movement or connection issues. `check_heartbeat_is_realistic` filters out these errors by ignoring impossible values. For example, it discards any reading that suggests a heart rate above 200 bpm (RR<300ms) or below 40 bpm (RR>1500ms), since that isn't realistic for a seated person. This prevents bad data from distorting the system's calculations.

In [7]:
def check_heartbeat_is_realistic(rr_value):
    """
    A RR interval that is unrealistically short or long is probably a sensor error rather than a real heartbeat. 
    This function returns True if the value looks like a real heartbeat, False if it should be ignored.
    """
    if rr_value < 300: # 300ms = 200 bpm (too fast for realistic resting heart rate)
        return False
    if rr_value > 1500: # 1500ms = 40 bpm (too slow)
        return False
    return True

### 7.2 `StressController`: the PID controller

This class acts as the system's memory. It remembers past data (accumulated error, last stress level, last timestamp) so the main program doesn't have to. On every heartbeat, it calculates a final reaction score between -100 and +100 through `calculate_pid_score`:

- Calculates the **proportional** part based on the user's *current* stress level
- Adds up the **integral** part to track *past* stress buildup (with limits to prevent it from growing too large)
- Calculates the **derivative** part based on *how fast* the stress is changing
- Combines these three parts and limits the final score to a range between `[-100, +100]`.

If the sensor disconnects for more than 2 seconds, it caps the time gap. This prevents the system from generating an exaggerated response when the connection is restored.

In [8]:
class StressController:
    """Manages the PID controller's memory so the main program doesn't have to track past data."""
    def __init__(self): # Setting controller memory to blank at start of a session
        self.accumulated_error = 0.0
        self.last_stress_level = 0.0
        self.last_timestamp = None
        self.rate_of_change = 0.0 # Stored on .self so write_monitor_data can read it after this function returns

    def calculate_pid_score(self, current_stress_level): # current_stress_level corresponds to error e(t)
        """The PID controller looks at the stress level and produces a score between -100 (very stressed) and +100 (very calm)."""
        current_time = time.monotonic() # Grabbing current time

        if self.last_timestamp is None:
            seconds_passed = 1.0 # First time, assume 1 second has passed
        else:
            seconds_passed = current_time - self.last_timestamp

        self.last_timestamp = current_time # Saving the current time for next time this function is called

        if seconds_passed > 2.0: # Capping the time at 2 seconds to prevent big jumps after pauses
            seconds_passed = 2.0

        p_reaction = KP * current_stress_level # Proportional: calculates how stressed user is right now

        self.accumulated_error += (current_stress_level * seconds_passed) # Running total: aking the current stress, multiplying it by how many seconds just passed (dt), and adding it to running total

        if self.accumulated_error > 50.0: # Limits past stress memory so the system can still react quickly when the user relaxes without becoming stuck/not responding; if this value becomes very high, multiplying it by a low Ki won't prevent the problem
            self.accumulated_error = 50.0
        if self.accumulated_error < -50.0:
            self.accumulated_error = -50.0

        i_reaction = KI * self.accumulated_error # Integral: adds up all stress detected so far, multiplied by time passed

        # Derivative: calculates how quickly the stress level is changing
        if seconds_passed > 0:
            rate_of_change = (current_stress_level - self.last_stress_level) / seconds_passed
        else:
            rate_of_change = 0.0 

        self.rate_of_change = rate_of_change # Stored on self so it can be read outside this function (e.g. by write_monitor_data)
        self.last_stress_level = current_stress_level # Saves the current stress reading so it has something to compare against when the next heartbeat arrives

        d_reaction = KD * rate_of_change

        final_score = p_reaction + i_reaction + d_reaction # Adding all three parts together to get the PID score

        if final_score > PID_max: # Ensure scores stays between PID_min and PID_max
            final_score = PID_max
        if final_score < PID_min:
            final_score = PID_min

        return final_score

### 7.3 `LightController`: translating score into light movement

This class manages the lighting output through two functions:

1. `convert_score_to_light_values` proportionally scales the PID score into specific brightness and colour temperature targets, based on the predefined "calm" and "stressed" limits.
2. `move_light_one_step` shifts the lighting toward these targets by a fixed amount per heartbeat. This prevents sudden light shifts.

In [9]:
class LightController:
    """This class remembers the current state of the lights so it can move them smoothly over time."""
    def __init__(self, start_brightness, start_colour_temp):
        self.current_brightness  = start_brightness
        self.current_colour_temp = start_colour_temp

    def convert_score_to_light_values(self, pid_score):
        """Takes the PID score (-100 to +100) and converts it into specific numbers for the light."""
        calm_percentage = (pid_score - PID_min) / (PID_max - PID_min) # Converting PID score to a percentage (0.0 = stressed, 1.0 = calm)

        if calm_percentage < 0.0:
            calm_percentage = 0.0
        if calm_percentage > 1.0:
            calm_percentage = 1.0

        brightness_range = brightness_calm - brightness_stressed 
        target_brightness = int(brightness_stressed + calm_percentage * brightness_range) # Calculating brightness: starts at stressed level, adds the calm amount

        colour_range = colour_temp_stressed - colour_temp_calm
        target_colour_temp = int(colour_temp_stressed - calm_percentage * colour_range) # Calculating colour temperature in the same way

        return target_brightness, target_colour_temp

    async def move_light_one_step(self, light, target_brightness, target_colour_temp):
        """Instead of shifting the light directly to a new value, this function moves it one small step at a time toward the target."""
        brightness_step = 25
        colour_step = 20

        if self.current_brightness < target_brightness:
            self.current_brightness = self.current_brightness + brightness_step # Moving brightness one step toward the target
            if self.current_brightness > target_brightness: # Preventing overshooting target value
                self.current_brightness = target_brightness
        elif self.current_brightness > target_brightness:
            self.current_brightness = self.current_brightness - brightness_step
            if self.current_brightness < target_brightness:
                self.current_brightness = target_brightness

        if self.current_colour_temp < target_colour_temp:
            self.current_colour_temp = self.current_colour_temp + colour_step # Moving temperature one step toward target
            if self.current_colour_temp > target_colour_temp:
                self.current_colour_temp = target_colour_temp
        elif self.current_colour_temp > target_colour_temp:
            self.current_colour_temp = self.current_colour_temp - colour_step
            if self.current_colour_temp < target_colour_temp:
                self.current_colour_temp = target_colour_temp

        await light.set_colour_temp(self.current_colour_temp) # Sending the updated values to the actual Hue light
        await light.set_brightness(self.current_brightness)


### 7.4 `update_music_mix`: the automated mixing desk

The music builds up in four distinct layers, fading in smoothly as the user's stress score improves from -100 to +100:

| Layer | Always on? | Fades in over |
|---|---|---|
| Drone | Yes, full volume throughout | - |
| Chords | No | -100 → -33 |
| Rhythm | No | -33 → +33 |
| Melody | No | +33 → +100 |

Instead of turning layers abruptly on or off, the system proportionally calculates the volume for each instrument based on the exact score. This ensures the music layers blend together seamlessly.

In [10]:
def update_music_mix(pid_score):
    """As the user calms down, it ensures more layers of music gradually fade in."""
    drone_max  = 1.0
    chords_max = 0.4
    rhythm_max = 0.8
    melody_max = 0.9

    ch1.set_volume(drone_max) # Layer 1: Drone always playing

    # Layer 2: Chords fade in from -100 to -33
    if pid_score > -100:
        fade_amount = (pid_score + 100) / 67.0 # zone width = 67; if pid_score=-100 (completely stressed), fade_amount=0; if pid_score=-33, fade_amount=1
        if fade_amount < 0.0:
            fade_amount = 0.0
        if fade_amount > 1.0:
            fade_amount = 1.0
        ch2.set_volume(fade_amount * chords_max)
    else:
        ch2.set_volume(0.0)

    # Layer 3: Rhythm fades in from -33 to +33
    if pid_score > -33:
        fade_amount = (pid_score + 33) / 66.0 # zone width = 66
        if fade_amount < 0.0:
            fade_amount = 0.0
        if fade_amount > 1.0:
            fade_amount = 1.0
        ch3.set_volume(fade_amount * rhythm_max)
    else:
        ch3.set_volume(0.0)

    # Layer 4: Melody fades in from +33 to +100
    if pid_score > 33:
        fade_amount = (pid_score - 33) / 67.0 # zone width = 67
        if fade_amount < 0.0:
            fade_amount = 0.0
        if fade_amount > 1.0:
            fade_amount = 1.0
        ch4.set_volume(fade_amount * melody_max)
    else:
        ch4.set_volume(0.0)


## 8. Heartbeat Processing Loop

This is the core of the system. It receives heartbeats from the sensor one by one (via `asyncio.Queue`) and processes them in two phases:

**Phase 1: Calibration.** The system collects the first 20 valid heartbeats to establish the user's personal resting baseline. If the user arrives already flustered (recording a heart rate over 95 bpm), the system overrides this and defaults to 75 bpm.

**Phase 2: Live operation.** For every heartbeat after calibration:

1. Tracks a fast (4-beat) and slow (8-beat) moving average of the heart rate.
2. Compares those two averages to generate a subtle light flicker that mimics the user's breathing patterns.
3. Compares the current heart rate against the baseline and feeds the difference into the PID controller to generate a reaction score.
4. Converts that score into updated lighting and music levels.
5. Logs all data into a JSON file, including how many ms the processing took. Tracking this speed provides evidence that the system functions efficiently in real-time.

In [11]:
async def process_heartbeats(queue, light, pid_controller, light_controller):
    """
    This function runs in a continuous loop for the entire time the program is running. It waits for a new heartbeat to arrive from the sensor, then processes it.

    - Phase 1 (Calibration): Collect the first 20 heartbeats to learn what is normal for this specific user.
    - Phase 2 (Live system): Use each new heartbeat to update the lights and music in real time.
    """

    baseline_rr = None
    calibration_list = []
    recent_heartbeats = []
    session_start_time = None

    start_log()

    print(f"Please sit still for {calibration_beats} heartbeats so the system can calibrate.")

    while True:

        heartbeat_data = await queue.get() # Waits until it receives a new heartbeat and then catches it for processing
        
        rr_interval = heartbeat_data[2][1] # bleakheart returns heartbeat_data as a tuple where position 2 contains the RR-interval information, and position 1 is the value in ms

        if not check_heartbeat_is_realistic(rr_interval):
            print(f"  Skipping unrealistic reading: {rr_interval}ms")
            queue.task_done()
            continue

        # Phase 1: Calibration
        if baseline_rr is None:
            calibration_list.append(rr_interval)
            beats_still_needed = calibration_beats - len(calibration_list)

            if beats_still_needed > 0:
                print(f"  Calibrating... {beats_still_needed} more beats needed")
                queue.task_done() # Tells the system it's finished with current heartbeat since unrealistic
                continue # Disregards following code and goes back to waiting for new heartbeat

            total = 0
            for beat in calibration_list:
                total += beat
            baseline_rr = total / len(calibration_list)

            bpm = 60000 / baseline_rr # Obtaining user heart rate in bpm
            stress_threshold_bpm  = 95
            fallback_baseline_bpm = 75

            if bpm > stress_threshold_bpm:
                baseline_rr = 60000 / fallback_baseline_bpm
                print(f"\nParticipant walked in at {bpm:.0f} bpm.")
                print(f"OVERRIDE: Forcing baseline to {fallback_baseline_bpm} bpm.")

            print(f"\nCalibration complete.")
            print(f"Your personal baseline: {baseline_rr:.1f}ms per beat")
            print(f"That is approximately {60000 / baseline_rr:.0f} beats per minute")
            print(f"\nThe system is now active.\n")

            # Writing calibration result to monitor file
            write_monitor_data({
                "calibrated": True, # Telling dashboard that calibration is done
                "baseline_flag": "HIGH" if bpm > 90 else ("MODERATE" if bpm > 80 else "NORMAL"),
                "baseline_rr": baseline_rr,
                "bpm": 0.0,
                "rr_avg": 0.0,
                "deviation": 0.0,
                "p_value": 0.0,
                "i_value": 0.0,
                "d_value": 0.0,
                "reaction_score": 0.0,
                "brightness": brightness_calm,
                "colour_temp": colour_temp_calm,
                "vol_drone": 0.0,
                "vol_chords": 0.0,
                "vol_rhythm": 0.0,
                "vol_melody": 0.0, # Setting values to 0s so dashboard starts with a clean state
            })

            update_music_mix(PID_min)

        # Phase 2: Live operation
        else:
            cycle_start = time.monotonic() # Captures time the system grabs the heartbeat

            recent_heartbeats.append(rr_interval)

            if len(recent_heartbeats) > beats_number_avg:
                recent_heartbeats.pop(0) # Deletes first item to ensure we have 8 beats

            if len(recent_heartbeats) < beats_number_avg:
                queue.task_done() # Jumps back at top of the loop for next beat
                continue

            # Calculating 8-beat average
            total = 0
            for beat in recent_heartbeats:
                total += beat
            average_rr = total / len(recent_heartbeats)

            # Calculating 4-beat average for breathing modulation
            breath = recent_heartbeats[-4:]
            total_breath = 0
            for unit in breath:
                total_breath += unit
            average_breath = total_breath / len(breath)

            # Breathing delta (difference between fast and slow average)
            breath_multiplier = 2.5 # Amplifying effect by 2.5x so it becomes more visible
            delta = (average_rr - average_breath) * breath_multiplier # Isolating how much the heart is speeding up or slowing down right now due to breathing
            if delta > 40: # Ensuring brightness doesnt shift 40 steps up or down due to breathing
                delta = 40
            elif delta < -40:
                delta = -40

            # Calculating stress level and PID score
            difference_from_baseline = average_rr - baseline_rr
            scaled_stress = (difference_from_baseline / rr_sensitivity) * 100.0 # Converting ms interval into a fraction, then transorming it into percentage
            reaction_score = pid_controller.calculate_pid_score(scaled_stress)

            # Converting PID score to light targets
            target_brightness, target_colour_temp = light_controller.convert_score_to_light_values(reaction_score)

            # Adding breathing delta to brightness only
            final_target_brightness = int(target_brightness + delta)
            if final_target_brightness > 254:
                final_target_brightness = 254
            elif final_target_brightness < 0:
                final_target_brightness = 0

            # Updating music/lights so volumes/colour temp+brightness are current before logging
            update_music_mix(reaction_score)

            await light_controller.move_light_one_step(light, final_target_brightness, target_colour_temp)

            # Stopping the timer, everything above has now completed
            cycle_end = time.monotonic()
            cycle_time_ms = (cycle_end - cycle_start) * 1000

            # Logging everything to CSV, all values now reflect this heartbeat
            if session_start_time is None: # If this is the first heartbeat
                session_start_time = time.monotonic()

            elapsed_seconds = time.monotonic() - session_start_time

            write_log_row( # Writing values to CSV
                time_s = elapsed_seconds,
                rr_avg = average_rr,
                baseline = baseline_rr,
                diff = difference_from_baseline,
                score = reaction_score,
                brightness = light_controller.current_brightness,
                ct = light_controller.current_colour_temp,
                v1 = ch1.get_volume(),
                v2 = ch2.get_volume(),
                v3 = ch3.get_volume(),
                v4 = ch4.get_volume(),
                cycle_ms = cycle_time_ms,
            )

            write_monitor_data({ # Updating JSON for live monitor
                "calibrated": True,
                "baseline_flag": "HIGH" if (60000 / baseline_rr) > 90 else ("MODERATE" if (60000 / baseline_rr) > 80 else "NORMAL"),
                "bpm": 60000 / average_rr,
                "rr_avg": average_rr,
                "baseline_rr": baseline_rr,
                "deviation": difference_from_baseline,
                "p_value": pid_controller.last_stress_level * KP,
                "i_value": pid_controller.accumulated_error * KI,
                "d_value": KD * pid_controller.rate_of_change, 
                "reaction_score": reaction_score,
                "brightness": light_controller.current_brightness,
                "colour_temp": light_controller.current_colour_temp,
                "vol_drone": ch1.get_volume(),
                "vol_chords": ch2.get_volume(),
                "vol_rhythm": ch3.get_volume(),
                "vol_melody": ch4.get_volume(),
            })

        queue.task_done()


## 9. System Initialisation

`main()` serves as the central coordinator for the application, executing the following setup sequence:

1. Connect to the Hue light over BLE and switch it on
2. Initialise the `StressController` and `LightController`
3. Connect to the Polar sensor and and begin streaming physiological data into a shared processing queue.
4. Launch `process_heartbeats` as a background task
5. Maintain the system state indefinitely until manually terminated by the researcher.

Device MAC addresses are hardcoded here (`sensor_address`, `light_address`). These should be updated to match the specific Polar strap and Hue light used in each session.

In [12]:
async def main():
    """Connects hardware, creates controllers, starts the loop."""

    sensor_address = "D4:8A:91:AC:B1:E9"
    light_address = "DE:0F:02:B0:58:1D"

    heartbeat_queue = asyncio.Queue() # Creating queue for incoming heartbeats

    print("Looking for the Hue light...")
    light_device = await BleakScanner.find_device_by_address(light_address) # Pausing and waiting until it finds the lights
    hue_light = HueBleLight(light_device)
    await hue_light.set_power(True) # Turning lights on if not already on
    print("Hue light connected.\n")

    my_pid_controller = StressController() # Creating new instances for PID and light controller
    my_light_controller = LightController(start_brightness = brightness_calm, start_colour_temp = colour_temp_calm)

    print("Looking for the Polar sensor...")
    async with BleakClient(sensor_address) as client: # Establishing connection with Polar
        print("Polar sensor connected.\n")

        heart_rate_monitor = HeartRate( # Creting HeartRate object (from bleakheart)
            client = client,
            queue = heartbeat_queue,
            unpack = True # Each item it delivers will be one individual heartbeat data
        )

        await heart_rate_monitor.start_notify() # Switchins sensor on, telling it to start streaming heartbeats
        print("Sensor is now streaming heartbeat data...\n")

        heartbeat_task = asyncio.create_task(
            process_heartbeats(heartbeat_queue, hue_light, my_pid_controller, my_light_controller) # Launchig heartbeat processing in background
        )

        def on_heartbeat_task_done(task): # Surfaces any crash instead of silently freezing the system
            if task.cancelled(): # When task is cancelled, it exits function quietly
                return
            error = task.exception()
            if error is not None: # Checks whether task ended due to an error
                print(f"\nHeartbeat processing crashed and stopped: {error}\n")

        heartbeat_task.add_done_callback(on_heartbeat_task_done) # When heartbeat_task finishes for any reason, it automatically calls function above      

        print("System is fully running\n")

        while True:
            await asyncio.sleep(3600) # Keeping system alive: pauses for one hour, then loops back and does the same thing over and over


## 10. Running the system

To launch the application, the `main()` function is called. If running as a standard Python script, this is executed using `asyncio.run(main())`. However, because Jupyter Notebooks operate within their own active event loop, the system must be launched using `await main()` directly to avoid runtime errors. Execution should only begin when the Polar sensor and Hue light are powered on and prepared for connection.

In [ ]:
await main()

Looking for the Hue light...
Hue light connected.

Looking for the Polar sensor...
Polar sensor connected.

Sensor is now streaming heartbeat data...

System is fully running

Session log started.

Please sit still for 20 heartbeats so the system can calibrate.
  Calibrating... 19 more beats needed
  Calibrating... 18 more beats needed
  Calibrating... 17 more beats needed
  Calibrating... 16 more beats needed
  Calibrating... 15 more beats needed
  Calibrating... 14 more beats needed
  Calibrating... 13 more beats needed
  Calibrating... 12 more beats needed
  Calibrating... 11 more beats needed
  Calibrating... 10 more beats needed
  Calibrating... 9 more beats needed
  Calibrating... 8 more beats needed
  Calibrating... 7 more beats needed
  Calibrating... 6 more beats needed
  Calibrating... 5 more beats needed
  Calibrating... 4 more beats needed
  Calibrating... 3 more beats needed
  Calibrating... 2 more beats needed
  Calibrating... 1 more beats needed

Calibration complete.
Yo